# Deteção de Malware com Machine Learning

**Objetivo:** desenvolver um modelo de Machine Learning capaz de classificar ficheiros executáveis como **Benignos (0)** ou **Malware (1)** utilizando características estruturais e estatísticas presentes no dataset.

Dataset utilizado: `Malware_and_benign_recognition.csv`.


## 1. Importação e carregamento dos dados

In [ ]:
# Estas bibliotecas leem os dados, fazem contas e mostram gráficos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# O CSV contém uma linha por ficheiro e uma coluna por característica.
df = pd.read_csv("Malware_and_benign_recognition.csv")
print("Dimensões:", df.shape)
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'Malware_and_benign_recognition.csv'

## 2. Análise inicial

O dataset utilizado nesta execução contém **1156 registos e 27 colunas**. A coluna `Malicious` é a variável-alvo. A coluna `File` identifica o ficheiro e não será usada como feature do modelo.


In [ ]:
# Estas verificações ajudam a perceber a qualidade e o equilíbrio do dataset.
print(df.info())
print("\nValores em falta:")
print(df.isna().sum().sum())

print("\nDuplicados:", df.duplicated().sum())

# Mostra quantos exemplos pertencem a cada classe do problema.
print("\nDistribuição da variável alvo:")
print(df["Malicious"].value_counts())

df["Malicious"].value_counts().plot(kind="bar")
plt.title("Distribuição: Benigno vs Malware")
plt.xlabel("Malicious")
plt.ylabel("Número de amostras")
plt.show()

## 3. Separação das features e do target

`Malicious` é o target:
- `0` = Benigno
- `1` = Malware

`File` é removida para evitar que o nome do ficheiro seja utilizado como informação preditiva.


In [ ]:
# Malicious é a resposta que queremos prever: 0 = benigno e 1 = malware.
TARGET = "Malicious"
# File identifica o ficheiro, mas não descreve o seu comportamento.
X = df.drop(columns=["File", TARGET])
y = df[TARGET]

print("Features:", X.shape[1])
print("Amostras:", X.shape[0])

## 4. Divisão entre treino e teste

É utilizada uma divisão de 80% para treino e 20% para teste, com `stratify=y` para manter aproximadamente a mesma distribuição das classes nos dois conjuntos.


In [ ]:
from sklearn.model_selection import train_test_split

# Guardamos 20% dos dados para uma avaliação final imparcial.
# A estratificação mantém uma proporção semelhante de benignos e malware.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 5. Modelos

Serão comparados quatro classificadores:
1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. Extra Trees

A comparação usa validação cruzada estratificada de 5 folds.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Cada modelo representa uma forma diferente de aprender a separar as classes.
models = {
    "Logistic Regression": Pipeline([
        # A normalização coloca as features numa escala comparável.
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, random_state=42, class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42, n_estimators=200
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, random_state=42, class_weight="balanced"
    )
}

# A validação cruzada cria cinco divisões e testa cada modelo em cada uma.
# Isto reduz a dependência de uma única divisão acidental dos dados.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    scores = cross_validate(
        model, X, y, cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
    )
    # Guardamos a média das cinco rondas para comparar os modelos.
    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(results)
display(results_df.sort_values("F1", ascending=False))

## 6. Avaliação no conjunto de teste

Depois da validação cruzada, cada modelo é treinado no conjunto de treino e avaliado no conjunto de teste que ficou separado.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

test_results = []
fitted_models = {}

for name, model in models.items():
    # Agora o modelo aprende apenas com o conjunto de treino.
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    # A probabilidade da classe 1 é usada para calcular a ROC-AUC.
    prob = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model

    # Calculamos várias métricas porque cada uma evidencia um tipo de desempenho.
    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_results_df = pd.DataFrame(test_results)
display(test_results_df)

## 7. Matriz de confusão

A matriz de confusão permite observar quantos benignos e malwares foram classificados corretamente e quantos foram confundidos.


In [ ]:
# Escolhemos o Extra Trees como modelo final para observar os seus erros.
final_model = fitted_models["Extra Trees"]
final_pred = final_model.predict(X_test)

# O relatório resume o desempenho por classe, incluindo os malwares encontrados.
print(classification_report(
    y_test, final_pred,
    target_names=["Benigno", "Malware"]
))

# A matriz mostra contagens de verdadeiros positivos, falsos positivos e restantes casos.
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred,
    display_labels=["Benigno", "Malware"]
)
plt.title("Matriz de Confusão - Extra Trees")
plt.show()

## 8. Importância das features

Para o modelo Extra Trees, podemos observar quais características tiveram maior importância na decisão do classificador.


In [ ]:
# O modelo atribui uma importância a cada feature usada pelas árvores.
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance.head(15))

# O gráfico facilita a comparação das 15 features com maior contributo.
importance.head(15).sort_values("Importance").plot(
    x="Feature", y="Importance", kind="barh", legend=False
)
plt.title("15 Features Mais Importantes")
plt.xlabel("Importância")
plt.show()

## 9. Conclusão

O projeto demonstra a aplicação de Machine Learning à deteção de malware através de classificação binária. Foram comparados quatro algoritmos e avaliados através de validação cruzada e de um conjunto de teste separado.

As métricas principais são Accuracy, Precision, Recall, F1 e ROC-AUC. Em cibersegurança, o Recall da classe Malware é particularmente relevante porque mede a capacidade de encontrar amostras maliciosas.

Os resultados obtidos neste dataset não devem ser interpretados automaticamente como desempenho em ficheiros reais fora do dataset. Para uma utilização real seriam necessários testes com amostras externas, controlo de data leakage, validação temporal e monitorização contínua.
